# DeepLabV3+ Implementation for Brain Tumor Segmentation

## Overview

This notebook implements the complete DeepLabV3+ architecture for brain tumor segmentation. DeepLabV3+ is a state-of-the-art semantic segmentation model that combines:

1. **Atrous Spatial Pyramid Pooling (ASPP)**: Captures multi-scale context information
2. **Encoder-Decoder Structure**: Refines segmentation details using skip connections
3. **ResNet50 Backbone**: Provides powerful feature extraction capabilities

### Learning Objectives

1. **Build ASPP Module**: Implement multi-scale feature extraction with atrous convolutions
2. **Construct Encoder-Decoder**: Create the full DeepLabV3+ architecture
3. **Handle Class Imbalance**: Apply Median Frequency Balancing (MFB) weights
4. **Implement Custom Loss Functions**: Combine Cross-Entropy and Dice Loss
5. **Train and Evaluate**: Train the model on brain tumor data

### Key Concepts

- **Atrous Convolution**: Dilated convolutions to increase receptive field
- **ASPP**: Parallel atrous convolutions with different dilation rates
- **Dice Loss**: Overlap-based loss function for segmentation
- **Median Frequency Balancing**: Class weight calculation for imbalanced data

In [ ]:
# Import required libraries
import os
import warnings
warnings.filterwarnings("ignore")

import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Conv2D, BatchNormalization, ReLU, Add, Input,
    UpSampling2D, Concatenate, AveragePooling2D
)
from tensorflow.keras import losses
from tensorflow.keras import callbacks
from sklearn.model_selection import train_test_split

print(f"TensorFlow version: {tf.__version__}")

In [ ]:
# GPU Configuration
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"Memory growth enabled for {len(gpus)} GPU(s)")
    except RuntimeError as e:
        print(f"Error: {e}")

## 1. Data Loading and Preprocessing

### Dataset Structure

The brain tumor segmentation dataset has the following structure:

```
Brain Tumor Segmentation Dataset/
├── image/
│   ├── 0/          # Class 0: No tumor
│   ├── 1/          # Class 1: Meningioma
│   ├── 2/          # Class 2: Glioma
│   └── 3/          # Class 3: Pituitary tumor
└── mask/
    ├── 0/          # Corresponding masks
    ├── 1/
    ├── 2/
    └── 3/
```

### Data Loading Strategy

1. Each image has a corresponding mask with `_m` suffix
2. Masks contain pixel values 0 (background) and 255 (tumor region)
3. Labels are based on the class directory name

In [ ]:
# Dataset configuration
DATASET_PATH = 'Brain Tumor Segmentation Dataset/'
IMG_SIZE = (256, 256)
BATCH_SIZE = 8
NUM_CLASSES = 4

In [ ]:
# Setup paths
image_base_path = os.path.join(DATASET_PATH, 'image')
mask_base_path = os.path.join(DATASET_PATH, 'mask')

# Get class directories
class_dirs = [d for d in os.listdir(image_base_path) 
              if os.path.isdir(os.path.join(image_base_path, d))]
class_dirs.sort()
print(f"Classes found: {class_dirs}")

In [ ]:
# Collect all image and mask paths
all_image_paths = []
all_mask_paths = []
all_labels = []

for class_dir in class_dirs:
    image_folder = os.path.join(image_base_path, class_dir)
    mask_folder = os.path.join(mask_base_path, class_dir)
    label = int(class_dir)
    
    for file_name in os.listdir(image_folder):
        if file_name.lower().endswith(('.png', '.jpg', '.jpeg', '.tif')):
            img_path = os.path.join(image_folder, file_name)
            base_name, ext = os.path.splitext(file_name)
            mask_filename = f"{base_name}_m{ext}"
            mask_path = os.path.join(mask_folder, mask_filename)
            
            if os.path.exists(mask_path):
                all_image_paths.append(img_path)
                all_mask_paths.append(mask_path)
                all_labels.append(label)

print(f"Total samples: {len(all_image_paths)}")

In [ ]:
# Train/Validation split
train_images, val_images, train_masks, val_masks, train_labels, val_labels = train_test_split(
    all_image_paths, all_mask_paths, all_labels,
    test_size=0.1, random_state=42, stratify=all_labels
)

print(f"Training samples: {len(train_images)}")
print(f"Validation samples: {len(val_images)}")

In [ ]:
# Check mask pixel values
all_unique_values = set()
for mask_path in all_mask_paths[:10]:
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    unique_values = np.unique(mask)
    all_unique_values.update(unique_values)

print(f"Unique pixel values in masks: {sorted(list(all_unique_values))}")

# Verify masks contain only 0 and 255
valid_masks = all([v in {0, 255} for v in all_unique_values])
print(f"All masks are binary (0 and 255): {valid_masks}")

## 2. Data Pipeline

### Data Augmentation Strategy

We apply the following augmentations to improve generalization:

1. **Photometric Augmentations**:
   - Random brightness adjustment (±5%)
   - Random contrast adjustment (95-105%)

2. **Preprocessing for ResNet50**:
   - `preprocess_input` from ResNet50 (converts to BGR, subtracts ImageNet mean)

In [ ]:
def load_image_and_mask(image_path, mask_path, label):
    """Load and preprocess image and mask."""
    # Load image
    img = tf.io.read_file(image_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    
    # Load and process mask
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    
    # Convert to class labels (0 for background, label for tumor)
    mask = tf.where(mask > 128, tf.cast(label, tf.uint8), tf.cast(0, tf.uint8))
    mask = tf.image.resize(mask, IMG_SIZE, method='nearest')
    
    return img, mask

In [ ]:
def augment_photometric(image, mask):
    """Apply photometric augmentations to the image only."""
    image = tf.image.random_brightness(image, max_delta=0.05)
    image = tf.image.random_contrast(image, lower=0.95, upper=1.05)
    image = tf.clip_by_value(image, 0.0, 255.0)
    return image, mask

In [ ]:
from tensorflow.keras.applications.resnet50 import preprocess_input

def final_preprocess_for_resnet(image, mask):
    """
    Apply ResNet50-specific preprocessing.
    
    Important: ResNet50 expects input values in a specific range.
    The preprocess_input function converts RGB to BGR and subtracts
    the ImageNet mean values.
    """
    image = preprocess_input(image)
    return image, mask

In [ ]:
# Create training dataset
train_dataset = tf.data.Dataset.from_tensor_slices((train_images, train_masks, train_labels))
train_dataset = train_dataset.map(load_image_and_mask, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.map(augment_photometric, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.map(final_preprocess_for_resnet, num_parallel_calls=tf.data.AUTOTUNE)
train_dataset = train_dataset.shuffle(buffer_size=1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# Create validation dataset
val_dataset = tf.data.Dataset.from_tensor_slices((val_images, val_masks, val_labels))
val_dataset = val_dataset.map(load_image_and_mask, num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.map(final_preprocess_for_resnet, num_parallel_calls=tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"Training dataset: {train_dataset}")
print(f"Validation dataset: {val_dataset}")

In [ ]:
# Verify preprocessing
for images_batch, masks_batch in train_dataset.take(1):
    print("Sample image stats after preprocessing:")
    image = images_batch[0].numpy()
    print(f"  Shape: {image.shape}")
    print(f"  Dtype: {image.dtype}")
    print(f"  Min: {image.min():.4f}")
    print(f"  Max: {image.max():.4f}")
    print(f"  Mean: {image.mean():.4f}")
    print("Note: Values are in ResNet50 preprocessed format (BGR, ImageNet mean subtracted)")

## 3. Class Imbalance Analysis

### Problem: High Class Imbalance

In medical image segmentation, the background often dominates the image. For example:
- Background pixels: ~95% of the image
- Tumor pixels: ~5% of the image

### Solution: Median Frequency Balancing (MFB)

MFB calculates class weights as the ratio of median frequency to class frequency:

```
weight_i = median_frequency / frequency_i
```

This gives higher weights to rare classes, balancing the loss contribution.

In [ ]:
# Calculate class frequencies
class_counts = {0: 0, 1: 0, 2: 0, 3: 0}
total_pixels = 0

for images_batch, masks_batch in train_dataset:
    for mask in masks_batch:
        mask_np = mask.numpy()
        total_pixels += mask_np.size
        unique, counts = np.unique(mask_np, return_counts=True)
        for u, c in zip(unique, counts):
            if u in class_counts:
                class_counts[u] += c

print(f"Total pixels: {total_pixels:,}")
print("\nClass frequencies:")
for cls, count in class_counts.items():
    freq = count / total_pixels
    print(f"  Class {cls}: {count:,} pixels ({freq:.2%})")

In [ ]:
# Median Frequency Balancing (MFB)
class_frequencies = {cls: count / total_pixels for cls, count in class_counts.items() if count > 0}
frequencies = list(class_frequencies.values())
median_frequency = np.median(frequencies)

mfb_weights = {}
for cls in class_counts:
    if cls in class_frequencies:
        mfb_weights[cls] = median_frequency / class_frequencies[cls]
    else:
        mfb_weights[cls] = 0.0

print("\nMedian Frequency Balancing weights:")
for cls, weight in mfb_weights.items():
    print(f"  Class {cls}: {weight:.4f}")

## 4. DeepLabV3+ Architecture Components

### 4.1 Convolution Block

A standard convolution block with batch normalization and ReLU activation.

In [ ]:
def convolution_block(
    block_input,
    num_filters=256,
    kernel_size=3,
    dilation_rate=1,
    padding="same",
    use_bias=True,
):
    """
    Standard convolution block: Conv2D -> BatchNorm -> ReLU
    
    Args:
        block_input: Input tensor
        num_filters: Number of filters in the convolution
        kernel_size: Size of the convolution kernel
        dilation_rate: Dilation rate for atrous convolution
        padding: Padding strategy ('same' or 'valid')
        use_bias: Whether to use bias in convolution
    """
    x = Conv2D(
        num_filters,
        kernel_size=kernel_size,
        dilation_rate=dilation_rate,
        padding=padding,
        use_bias=use_bias,
    )(block_input)
    x = BatchNormalization()(x)
    x = ReLU()(x)
    return x

### 4.2 Atrous Spatial Pyramid Pooling (ASPP)

ASPP applies parallel atrous convolutions with different dilation rates to capture multi-scale context:

```
Input
  ├── Average Pooling (Global) → Upsample → Conv
  ├── Conv (rate=1)
  ├── Conv (rate=6)
  ├── Conv (rate=12)
  └── Conv (rate=18)
       ↓ Concatenate
       Conv (1x1)
       Output
```

**Why different dilation rates?**
- Rate 1: Captures local context
- Rate 6: Captures medium-range context
- Rate 12: Captures larger context
- Rate 18: Captures global context
- Global Pooling: Captures entire image context

In [ ]:
def AtrousSpatialPyramidPooling(aspp_input):
    """
    Atrous Spatial Pyramid Pooling module.
    
    This module captures multi-scale context information using parallel
    atrous convolutions with different dilation rates.
    """
    dims = aspp_input.shape
    
    # Global average pooling branch
    x = AveragePooling2D(pool_size=(dims[1], dims[2]))(aspp_input)
    out_pool = UpSampling2D(size=(dims[1], dims[2]), interpolation="bilinear")(x)
    
    # Parallel atrous convolutions with different dilation rates
    out_1 = convolution_block(aspp_input, kernel_size=1, dilation_rate=1)
    out_6 = convolution_block(aspp_input, kernel_size=3, dilation_rate=6)
    out_12 = convolution_block(aspp_input, kernel_size=3, dilation_rate=12)
    out_18 = convolution_block(aspp_input, kernel_size=3, dilation_rate=18)
    
    # Concatenate all branches
    x = Concatenate(axis=-1)([out_pool, out_1, out_6, out_12, out_18])
    output = convolution_block(x, kernel_size=1)
    
    return output

### 4.3 DeepLabV3+ Encoder-Decoder Architecture

The complete DeepLabV3+ architecture consists of:

1. **Encoder**:
   - ResNet50 backbone (modified with atrous convolutions)
   - ASPP module for multi-scale context

2. **Decoder**:
   - Upsample ASPP output 4x
   - Concatenate with low-level features from conv2_block3_out
   - Apply additional convolutions
   - Final upsample to original resolution

### Output Stride (OS)

We use OS=16 to balance accuracy and memory usage:
- Feature map reduced by 16x (256x256 → 16x16)
- Good compromise between detail and context

In [ ]:
def DeeplabV3_OS16(image_size, num_classes):
    """
    Build DeepLabV3+ model with Output Stride = 16.
    
    Args:
        image_size: Input image size (height, width)
        num_classes: Number of segmentation classes
    
    Returns:
        Keras Model
    """
    model_input = keras.Input(shape=(image_size, image_size, 3))
    
    # Load pre-trained ResNet50 backbone
    base_model = keras.applications.ResNet50(
        weights='imagenet',
        include_top=False,
        input_tensor=model_input
    )
    base_model.trainable = False
    
    # Extract features
    x = base_model.get_layer("conv4_block6_out").output  # High-level features
    low_level_features = base_model.get_layer("conv2_block3_out").output  # Low-level features
    
    # Custom conv5 blocks with atrous convolutions (no downsampling)
    for i in range(1, 4):
        shortcut = x
        
        if i == 1:
            # First block: has projection shortcut
            x = Conv2D(512, (1, 1), strides=(1, 1), padding="same",
                       name=f"custom_conv5_block{i}_1_conv")(x)
            x = BatchNormalization(name=f"custom_conv5_block{i}_1_bn")(x)
            x = ReLU(name=f"custom_conv5_block{i}_1_relu")(x)
            
            x = Conv2D(512, (3, 3), strides=(1, 1), padding="same",
                       dilation_rate=2, name=f"custom_conv5_block{i}_2_conv")(x)
            x = BatchNormalization(name=f"custom_conv5_block{i}_2_bn")(x)
            x = ReLU(name=f"custom_conv5_block{i}_2_relu")(x)
            
            x = Conv2D(2048, (1, 1), strides=(1, 1), padding="same",
                       name=f"custom_conv5_block{i}_3_conv")(x)
            x = BatchNormalization(name=f"custom_conv5_block{i}_3_bn")(x)
            
            # Projection shortcut
            shortcut = Conv2D(2048, (1, 1), strides=(1, 1), padding="same",
                             name=f"custom_conv5_block{i}_0_conv")(shortcut)
            shortcut = BatchNormalization(name=f"custom_conv5_block{i}_0_bn")(shortcut)
        else:
            # Subsequent blocks: standard residual
            x = Conv2D(512, (1, 1), strides=(1, 1), padding="same",
                       name=f"custom_conv5_block{i}_1_conv")(x)
            x = BatchNormalization(name=f"custom_conv5_block{i}_1_bn")(x)
            x = ReLU(name=f"custom_conv5_block{i}_1_relu")(x)
            
            x = Conv2D(512, (3, 3), strides=(1, 1), padding="same",
                       dilation_rate=2, name=f"custom_conv5_block{i}_2_conv")(x)
            x = BatchNormalization(name=f"custom_conv5_block{i}_2_bn")(x)
            x = ReLU(name=f"custom_conv5_block{i}_2_relu")(x)
            
            x = Conv2D(2048, (1, 1), strides=(1, 1), padding="same",
                       name=f"custom_conv5_block{i}_3_conv")(x)
            x = BatchNormalization(name=f"custom_conv5_block{i}_3_bn")(x)
        
        x = Add(name=f"custom_conv5_block{i}_add")([x, shortcut])
        x = ReLU(name=f"custom_conv5_block{i}_out")(x)
    
    encoder_output = x
    
    # ASPP module
    x = AtrousSpatialPyramidPooling(encoder_output)
    
    # Decoder
    input_a = UpSampling2D(size=(4, 4), interpolation="bilinear")(x)
    input_b = convolution_block(low_level_features, num_filters=48, kernel_size=1)
    
    x = Concatenate(axis=-1)([input_a, input_b])
    x = convolution_block(x, num_filters=256)
    x = convolution_block(x, num_filters=256)
    x = UpSampling2D(size=(4, 4), interpolation="bilinear")(x)
    
    # Final prediction layer
    model_output = Conv2D(num_classes, kernel_size=(1, 1), padding="same")(x)
    
    return Model(inputs=model_input, outputs=model_output)

In [ ]:
# Build the model
model = DeeplabV3_OS16(image_size=IMG_SIZE[0], num_classes=NUM_CLASSES)
model.summary()

## 5. Loss Functions

### 5.1 Dice Loss

Dice loss is based on the Dice coefficient:

```
Dice = 2 * |A ∩ B| / (|A| + |B|)
Dice Loss = 1 - Dice
```

**Advantages over Cross-Entropy:**
- Handles class imbalance better
- Focuses on overlap rather than pixel-wise accuracy
- More sensitive to the shape of the prediction

**Disadvantages:**
- Noisy gradients (sensitive to small changes)
- May converge slowly
- Can get stuck in local minima

### 5.2 Combined Loss

We combine Cross-Entropy and Dice Loss:

```
Combined Loss = Cross-Entropy + Dice Loss
```

**Why combine?**
- Cross-Entropy: Smooth gradients, stable training
- Dice Loss: Handles imbalance, focuses on shape
- Combined: Best of both worlds

In [ ]:
def dice_loss(y_true, y_pred, smooth=1e-6):
    """
    Calculate Dice loss for segmentation.
    
    Args:
        y_true: Ground truth labels (sparse, shape: [batch, h, w])
        y_pred: Model predictions (logits, shape: [batch, h, w, num_classes])
        smooth: Small value to avoid division by zero
    
    Returns:
        Dice loss (1 - Dice coefficient)
    """
    # Apply softmax to predictions
    y_pred_probs = tf.nn.softmax(y_pred, axis=-1)
    
    # Convert sparse labels to one-hot
    y_true_one_hot = tf.one_hot(tf.cast(y_true, tf.int32), depth=4, axis=-1)
    y_true_one_hot = tf.squeeze(y_true_one_hot, axis=-2)
    
    # Calculate intersection and union
    intersection = tf.reduce_sum(y_true_one_hot * y_pred_probs, axis=[1, 2])
    sum_true = tf.reduce_sum(y_true_one_hot, axis=[1, 2])
    sum_pred = tf.reduce_sum(y_pred_probs, axis=[1, 2])
    
    # Dice coefficient
    dice_coefficient = (2. * intersection) / (sum_true + sum_pred + smooth)
    mean_dice = tf.reduce_mean(dice_coefficient)
    
    # Dice loss = 1 - Dice coefficient
    return 1. - mean_dice

def combined_loss(y_true, y_pred):
    """
    Combined loss: Cross-Entropy + Dice Loss.
    
    This combines the advantages of both losses:
    - Cross-Entropy: Smooth gradients, stable training
    - Dice Loss: Handles class imbalance, focuses on shape
    """
    scce = losses.sparse_categorical_crossentropy(y_true, y_pred, from_logits=True)
    dice = dice_loss(y_true, y_pred)
    return scce + dice

## 6. Model Compilation and Training

### Training Configuration

- **Optimizer**: Adam (learning rate = 1e-4)
- **Loss**: Combined loss (Cross-Entropy + Dice)
- **Metrics**: Accuracy, Mean IoU
- **Class Weights**: MFB weights for handling imbalance
- **Callbacks**: ModelCheckpoint (save best model)

In [ ]:
# Compile model
sparse_mean_iou = tf.keras.metrics.MeanIoU(num_classes=NUM_CLASSES, sparse_y_pred=False)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss=combined_loss,
    metrics=["accuracy", sparse_mean_iou]
)

In [ ]:
# Setup callbacks
callbacks = [
    callbacks.ModelCheckpoint(
        './bestmodel.keras',
        monitor="val_loss",
        verbose=1,
        save_best_only=True
    ),
]

print("Training configuration:")
print(f"  - Learning rate: 1e-4")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Epochs: 20")
print(f"  - Loss: Combined (Cross-Entropy + Dice)")
print(f"  - Class weights: MFB")

In [ ]:
# Train the model
history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=20,
    class_weight=mfb_weights,
    callbacks=callbacks
)

## 7. Training Visualization

In [ ]:
plt.figure(figsize=(20, 5))

plt.subplot(1, 3, 1)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss')
plt.legend()
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(history.history['mean_io_u'], label='Training Mean IoU')
plt.plot(history.history['val_mean_io_u'], label='Validation Mean IoU')
plt.title('Mean IoU')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 8. Evaluation and Prediction

In [ ]:
# Load best model
custom_objects = {"combined_loss": combined_loss}
best_model = tf.keras.models.load_model(
    './bestmodel.keras',
    custom_objects=custom_objects
)
print("Loaded best model")

In [ ]:
def predict_and_visualize(model, image_path, mask_path, label,
                         img_size=(256, 256), confidence_threshold=0.9):
    """
    Predict and visualize segmentation results.
    
    Args:
        model: Trained segmentation model
        image_path: Path to input image
        mask_path: Path to ground truth mask
        label: Class label
        img_size: Image size for prediction
        confidence_threshold: Minimum confidence for prediction
    """
    # Load and preprocess image
    img = tf.io.read_file(image_path)
    img = tf.image.decode_png(img, channels=3)
    img_resized = tf.image.resize(img, img_size)
    img_for_display = img_resized
    
    # Preprocess for ResNet50
    img_preprocessed = preprocess_input(img_resized)
    img_for_prediction = tf.expand_dims(img_preprocessed, axis=0)
    
    # Predict
    prediction = model.predict(img_for_prediction)
    max_probs = np.max(prediction, axis=-1)
    predicted_labels = np.argmax(prediction, axis=-1)
    
    # Apply confidence threshold
    predicted_mask = np.where(max_probs < confidence_threshold, 0, predicted_labels)
    predicted_mask = np.squeeze(predicted_mask)
    
    # Load ground truth mask
    _, true_mask_tensor = load_image_and_mask(image_path, mask_path, label)
    true_mask = np.squeeze(true_mask_tensor.numpy())
    
    # Visualize
    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.title("Original Image")
    plt.imshow(img_for_display / 255.0)
    plt.axis('off')

    plt.subplot(1, 3, 2)
    plt.title("True Mask")
    plt.imshow(true_mask, cmap='jet', vmin=0, vmax=3)
    plt.axis('off')

    plt.subplot(1, 3, 3)
    plt.title(f"Predicted Mask (Threshold={confidence_threshold})")
    plt.imshow(predicted_mask, cmap='jet', vmin=0, vmax=3)
    plt.axis('off')

    plt.tight_layout()
    plt.show()

    print(f"Label: {label}")
    print(f"Unique values in True Mask: {np.unique(true_mask)}")
    print(f"Unique values in Predicted Mask: {np.unique(predicted_mask)}")

In [ ]:
# Test prediction on a sample
sample_index = 1500
sample_image_path = train_images[sample_index]
sample_mask_path = train_masks[sample_index]
sample_label = train_labels[sample_index]

predict_and_visualize(best_model, sample_image_path, sample_mask_path, sample_label)

In [ ]:
# Evaluate on training set
train_metrics = best_model.evaluate(train_dataset, return_dict=True, verbose=0)
print("\nTraining Set Metrics:")
for key, value in train_metrics.items():
    print(f"  {key}: {value:.4f}")

In [ ]:
# Evaluate on validation set
val_metrics = best_model.evaluate(val_dataset, return_dict=True, verbose=0)
print("\nValidation Set Metrics:")
for key, value in val_metrics.items():
    print(f"  {key}: {value:.4f}")

## 9. Summary and Key Takeaways

### What We've Learned

1. **DeepLabV3+ Architecture**:
   - ASPP module for multi-scale context
   - Encoder-decoder structure with skip connections
   - ResNet50 backbone with atrous convolutions

2. **Data Handling**:
   - Loading and preprocessing brain tumor data
   - Applying augmentation for better generalization
   - Handling class imbalance with MFB weights

3. **Loss Functions**:
   - Dice Loss for handling class imbalance
   - Combined loss for stable training
   - Importance of gradient behavior

4. **Training**:
   - Using pre-trained weights for transfer learning
   - Monitoring training with callbacks
   - Evaluating with appropriate metrics

### Next Steps

1. **Experiment with different backbones** (EfficientNet, Inception)
2. **Try different output strides** (OS=8 for better detail)
3. **Apply post-processing** (CRF refinement)
4. **Use more advanced augmentation** (geometric transformations)
5. **Hyperparameter tuning** (learning rate, batch size)

### Key Insights

- **Pre-trained models** significantly improve performance
- **Class imbalance** must be addressed with appropriate techniques
- **Dice Loss** is more suitable for segmentation than Cross-Entropy alone
- **ASPP** provides multi-scale context that's crucial for segmentation